# 05 — Routing module test

Testing the new reusable `src/routing/pathfinding.py` module (`find_accessible_route`) on real data (Downtown Toronto), instead of redefining `accessibility_weight` by hand in every notebook.

### Download the area and import the real module

Same tag-whitelist fix as before. This time `accessibility_weight` isn't redefined here — it's imported from `src/graph/weighting.py` via the new `find_accessible_route` helper.

In [1]:
import sys
sys.path.append("..")

import osmnx as ox
import networkx as nx

from src.graph.tag_filters import find_edges_with_tag
from src.routing.pathfinding import find_accessible_route

ox.settings.cache_folder = "cache"
ox.settings.useful_tags_way += ["wheelchair", "surface"]
ox.settings.useful_tags_node += ["kerb", "crossing", "tactile_paving"]
ox.settings.overpass_url = "https://overpass.kumi.systems/api/interpreter"

G_big = ox.graph_from_place("Downtown Toronto, Toronto, Ontario, Canada", network_type="walk")
print(G_big.number_of_nodes(), "nodes")
print(G_big.number_of_edges(), "edges")

14123 nodes
39170 edges


### Run the real routing test through the module

Same scenario as notes/04: pick a real `wheelchair: no` edge, set source/target to its two endpoints, and compare the plain-length route against `find_accessible_route`'s output — but this time through one clean function call instead of hand-written weighting/attribute-setting code.

In [2]:
no_access_edges = find_edges_with_tag(G_big, "wheelchair", "no")
u_bad, v_bad, k_bad = no_access_edges[0]
print("testing around edge:", u_bad, "->", v_bad)

path_by_length = nx.shortest_path(G_big, source=u_bad, target=v_bad, weight="length")
path_by_acc, cost_by_acc = find_accessible_route(G_big, u_bad, v_bad)

print("shortest by length:", path_by_length)
print("shortest by acc_weight:", path_by_acc, " cost =", cost_by_acc)
print("same route?", path_by_length == path_by_acc)

testing around edge: 400707035 -> 516949102
shortest by length: [400707035, 516949102]
shortest by acc_weight: [400707035, 3590076075, 3590076072, 6412492716, 516949102]  cost = 198.50924529201063
same route? False
